# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
import os
os.environ["JAVA_HOME"] = "/home/trar3243/miniforge3/envs/spark_env"

from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [ ]:
spark = (
    SparkSession.builder
    .appName("patent-analysis")
    .master("local[*]")
    .config("spark.driver.memory", "8g") # was running into java heap errors 
    .config("spark.executor.memory", "8g")
    .getOrCreate()
)

from pathlib import Path

checkpoint_dir = Path("./spark_checkpoint").resolve()
checkpoint_dir.mkdir(parents=True, exist_ok=True)

spark.sparkContext.setCheckpointDir(checkpoint_dir.as_uri())

print(spark.sparkContext.getCheckpointDir())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 16:43:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


file:/home/trar3243/repos/datacenter_scale/lab4-pyspark-patent/spark_checkpoint/597a05f4-7672-4804-941c-850efd4dc342


Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows


In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

Your job is to augment the data in pat63_99.txt to include a column indicating the number of patents cited that originate from the same state. Obviously, this data can only be calculated for patents that have originating state information (and thus, only those from the US) and only for cited patents that provide that information.
You will report the ten patents that have the most self-state citations sorted in descending order. 
There are some complications:

Not all patents in the 'cited' table are in the 'patent' table
Not all patents cite other patents
Not all patents are cited by other patents
Lastly, the NaN/Null value used by PySpark makes sorting values involving Nan/Null and numeric values problematic; you're best filtering out the null values and then sorting or replacing those values with meaningful values.



In [ ]:
# tables: citations and patents 
from pyspark.sql import functions as F

# goal: get patent alongside count, we will join back to patent later 
counts = patents.join(
        citations, on=patents.PATENT == citations.CITING, how="inner" # join to citations as "CITING" to figure out which patent his patent cites 
    ).alias(
        "first"
    ).join(
        patents.alias("second"), on= (col("first.cited") == col("second.patent")) & (col("first.postate") == col("second.postate")) # join back to patents to get all the patent information for what patent each orig patent cites 
    ).filter(
        (col("first.postate").isNotNull()) & (col("first.postate") != "") # filter out null and empty states 
    ).select(
        col("first.patent").alias("patent_to_drop") # select only the "citing" patent 
    ).groupBy(
        col("patent_to_drop") # group by the patent because each citing patent now has multiple record for each in-state cited patent 
    ).agg(
        F.count("*").alias("SAME_STATE") # get count 
    ).checkpoint(eager=True) # checkpoint for ease of use later 



In [ ]:
full = patents.join(
    counts, on=patents.PATENT == counts.patent_to_drop, how="inner" # join back to the original patent thing to get the rest of the info. Reason to do this is so that we don't have to group by a bunch of columns 
    ).orderBy(
        col("SAME_STATE").desc() # order by number of same state citing 
    ).drop("patent_to_drop") # drop the extra join column 

full.show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|       103|
|6008204| 1999|14606|   1998| 